# Frases para encontrar en SEC EDGAR los contratos de arrendamiento de ALeaseBERT

#### Cómo se eligen las frases

Para cada contrato se sacan **4 frases de 12 palabras seguidas**:

1. Solo se miran los párrafos largos (22 palabras o más, y menos de 2.000 caracteres). Son texto de
   cláusulas, no títulos, firmas ni tablas, así que su redacción es propia de cada contrato.
2. De cada párrafo se toman las 12 palabras **del centro**. El principio de un párrafo suele ser un
   número o una fórmula repetida ("The Tenant shall…"); el centro es más específico.
3. De todos esos candidatos se cogen 4 **repartidos a lo largo del contrato** (uno de cada cuarto).
   Si las 4 frases, de partes distintas, llevan al mismo documento de EDGAR, no es casualidad.
4. Se quedan solo las letras: se quitan puntuación y números. El buscador de EDGAR ignora la
   puntuación, y así la frase no depende de cómo se copiaron comas, comillas o cifras.

12 palabras es bastante para que la frase sea única entre millones de documentos, y bastante corto
para que una pequeña diferencia de copia en otra parte del párrafo no la estropee.

In [1]:
import html
import json
import re
import urllib.parse
from pathlib import Path

# La raíz del repositorio: la primera carpeta hacia arriba que contiene infra/.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'infra').is_dir():
    ROOT = ROOT.parent
DATASET_DIR = ROOT.parent / 'datasets' / 'annotated_dataset'
POOL = DATASET_DIR / 'pool'
OUT = DATASET_DIR / 'edgar_search_phrases.json'

PHRASES_PER_DOC = 4
WORDS = 12

In [2]:
def paragraphs(path):
    """Las líneas del contrato, una por cada <p> del .plain.html."""
    s = open(path, encoding='utf-8', errors='replace').read()
    return [html.unescape(re.sub(r'<[^>]+>', '', p)).strip() for p in re.findall(r'<p id=[^>]*>(.*?)</p>', s, re.S)]


def phrases(paras):
    """Las 12 palabras del centro de cada párrafo largo, y de ellas 4 repartidas por el contrato."""
    cands = []
    for p in paras:
        words = re.findall(r'[A-Za-z]+', p)
        if len(words) >= WORDS + 10 and len(p) < 2000:
            mid = len(words) // 2 - WORDS // 2
            cands.append(' '.join(words[mid : mid + WORDS]))
    if not cands:
        return []
    step = max(1, len(cands) // PHRASES_PER_DOC)
    return cands[::step][:PHRASES_PER_DOC]


def edgar_search_url(phrase):
    """La búsqueda en la web de EDGAR, con la frase entre comillas y sin límite de fechas."""
    return 'https://www.sec.gov/edgar/search/#/q=' + urllib.parse.quote(f'"{phrase}"') + '&dateRange=all'

## Las frases de cada contrato anotado

In [3]:
rows = []
for ann in sorted(POOL.glob('*.ann.json')):
    stem = ann.name[: -len('.ann.json')]
    text = POOL / f'{stem}.plain.html'
    rows.append({
        'ann_json': ann.name,
        'plain_html': text.name,
        'contract': stem.split('-', 1)[1],
        'phrases': [{'text': p, 'edgar_search': edgar_search_url(p)} for p in phrases(paragraphs(text))],
    })

OUT.write_text(json.dumps(rows, indent=1, ensure_ascii=False))

by_count = {}
for r in rows:
    by_count[len(r['phrases'])] = by_count.get(len(r['phrases']), 0) + 1
print('contratos anotados:', len(rows))
for n in sorted(by_count, reverse=True):
    print(f'  con {n} frases: {by_count[n]}')
print('sin ninguna frase (no tienen párrafos largos):', [r['contract'] for r in rows if not r['phrases']])
print('guardado en', OUT)

contratos anotados: 179
  con 4 frases: 173
  con 3 frases: 2
  con 2 frases: 1
  con 1 frases: 1
  con 0 frases: 2
sin ninguna frase (no tienen párrafos largos): ['lease_579', 'lease_contract_203']
guardado en /home/sante/Documents/FGV/me/datasets/annotated_dataset/edgar_search_phrases.json


#### Ejemplo: `lease_contract_100`

Cada frase va con su enlace de búsqueda. Al abrirlo, la web de EDGAR ya tiene la frase entre
comillas y el rango de fechas en «All (since 2001)»; por defecto solo busca en los últimos 5 años.

In [4]:
example = next(r for r in rows if r['contract'] == 'lease_contract_100')
print(json.dumps(example, indent=1, ensure_ascii=False))

{
 "ann_json": "a.1kButNZQjWunDCwztKExV.Cwsu-lease_contract_100.ann.json",
 "plain_html": "a.1kButNZQjWunDCwztKExV.Cwsu-lease_contract_100.plain.html",
 "contract": "lease_contract_100",
 "phrases": [
  {
   "text": "terms based on China Contract Law China Real Estate Administrative Law Shenzhen",
   "edgar_search": "https://www.sec.gov/edgar/search/#/q=%22terms%20based%20on%20China%20Contract%20Law%20China%20Real%20Estate%20Administrative%20Law%20Shenzhen%22&dateRange=all"
  },
  {
   "text": "the land use fee and the taxes lease management fee that generated",
   "edgar_search": "https://www.sec.gov/edgar/search/#/q=%22the%20land%20use%20fee%20and%20the%20taxes%20lease%20management%20fee%20that%20generated%22&dateRange=all"
  },
  {
   "text": "any reconstruction renovation or decoration of the leased property should be decided",
   "edgar_search": "https://www.sec.gov/edgar/search/#/q=%22any%20reconstruction%20renovation%20or%20decoration%20of%20the%20leased%20property%20should%20be